# Projeto #2: Marketing Campaign Analysis

## Projeto #2: Marketing Campaign Analysis

**Domínio:** Telemarketing bancário (campanhas reais de call center)
**Pergunta:** Qual campanha converte? Por quê? Como otimizar o próximo ciclo?
**Conceitos cobertos:** Prompt Engineering (S1), RAG + Semantic Caching (S3-4),
Tool Use paralelo (S5), Multi-Agent (Analyzer + Recommender) (S6), LangGraph
com branch condicional (S6-7), Extended Thinking (S7), Observability de
latência (S8)

**Sobre os dados:** o plano original descrevia "1000 campanhas digitais
fictícias" (search/social/display/email, CTR/CPA). Não existe dataset
público real nesse formato exato — performance de campanha digital é dado
comercial sensível. Em vez de forçar um dataset real dentro de um schema
fictício, adaptamos honestamente pro domínio real disponível: o
[UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing)
(Moro et al., 2014) — 41.188 contatos telefônicos reais de uma campanha de
telemarketing bancário português, dos quais amostramos 10.000. Agregamos por
(canal, mês) pra formar "campanhas" — os conceitos de arquitetura (RAG, Tool
Use, Multi-Agent, LangGraph, Cache Semântico) são os mesmos; a métrica virou
**taxa de conversão** (assinou o produto ou não) em vez de CTR/CPA.

**Sobre realismo:** chama a **API real da Anthropic** quando
`ANTHROPIC_API_KEY` está no ambiente, com fallback determinístico sem ela.

In [1]:
!pip install -q langgraph pydantic anthropic pandas

import os
import random
import time
from typing import Literal, Optional
from pydantic import BaseModel, Field

### 1. Schema + dados reais agregados (substituem o CSV fictício de 1000 campanhas)

**Por que esta classe existe:** `CampaignRecord` normaliza os dados
agregados (contatos, sucessos, duração média da ligação) e expõe
`conversion_rate`/`cost_per_contact_estimate` como propriedades calculadas
— o custo é estimado (o dataset real não tem valor em $ por ligação; usamos
uma premissa documentada de $2/minuto de operador), mas contatos, sucessos e
duração são 100% reais.

In [2]:
class CampaignRecord(BaseModel):
    campaign_id: str
    channel: Literal["cellular", "telephone"]
    n_contacts: int
    n_success: int
    avg_duration_sec: float

    @property
    def conversion_rate(self) -> float:
        return round(self.n_success / max(self.n_contacts, 1), 4)

    @property
    def cost_per_success(self) -> float:
        """Custo estimado por conversão. Premissa documentada: $2/minuto de
        operador (o dataset real não inclui custo em $)."""
        total_minutes = self.avg_duration_sec * self.n_contacts / 60
        cost = total_minutes * 2
        return round(cost / max(self.n_success, 1), 2)

import pandas as pd

DATASET_PATH = "../../datasets/bank_marketing_campaigns.csv"

def load_real_campaigns(path: str = DATASET_PATH) -> list[CampaignRecord]:
    df = pd.read_csv(path)
    df["success"] = (df["y"] == "yes").astype(int)
    agg = df.groupby(["contact", "month"]).agg(
        n_contacts=("y", "count"),
        n_success=("success", "sum"),
        avg_duration=("duration", "mean"),
    ).reset_index()
    return [
        CampaignRecord(
            campaign_id=f"{row.contact}_{row.month}",
            channel=row.contact,
            n_contacts=int(row.n_contacts),
            n_success=int(row.n_success),
            avg_duration_sec=round(float(row.avg_duration), 1),
        )
        for row in agg.itertuples()
    ]

def generate_campaigns(n: int = 15, seed: int = 7) -> list[CampaignRecord]:
    """Fallback sintético — só usado se o CSV real não for encontrado."""
    random.seed(seed)
    out = []
    for i in range(n):
        n_contacts = random.randint(50, 600)
        n_success = int(n_contacts * random.uniform(0.03, 0.6))
        out.append(CampaignRecord(
            campaign_id=f"camp_{i:03d}",
            channel=random.choice(["cellular", "telephone"]),
            n_contacts=n_contacts,
            n_success=n_success,
            avg_duration_sec=round(random.uniform(200, 350), 1),
        ))
    return out

try:
    campaigns = load_real_campaigns()
    print(f"✓ {len(campaigns)} campanhas REAIS agregadas (UCI Bank Marketing, por canal+mês)")
except FileNotFoundError:
    campaigns = generate_campaigns()
    print(f"⚠️  datasets/bank_marketing_campaigns.csv não encontrado — usando {len(campaigns)} campanhas sintéticas")
print(campaigns[0])

✓ 20 campanhas REAIS agregadas (UCI Bank Marketing, por canal+mês)
campaign_id='cellular_apr' channel='cellular' n_contacts=599 n_success=122 avg_duration_sec=284.1


**Resultado esperado:** `✓ 20 campanhas REAIS agregadas...` — 20 grupos
reais (2 canais × até 10 meses), com taxa de conversão real variando de
~3.6% a ~64% entre grupos (a mediana real é ~16%).

### 2. Semantic cache (Semana 7) — evita reprocessar campanhas parecidas

**Por que esta função existe:** cache "by meaning" ao invés de "by exact
match" — duas campanhas com conversion_rate/custo parecidos no mesmo canal
caem no mesmo `cache_key` e reusam o resultado. Numa implementação real,
`cache_key` seria substituído por um embedding + busca de vizinho mais
próximo; aqui, o bucket arredondado já demonstra o princípio sem precisar
de um vector DB.

In [3]:
_semantic_cache: dict[tuple, dict] = {}

def cache_key(c: CampaignRecord) -> tuple:
    return (c.channel, round(c.conversion_rate, 1), round(c.cost_per_success / 20) * 20)

def cached_analysis(c: CampaignRecord, compute_fn):
    key = cache_key(c)
    if key in _semantic_cache:
        return {**_semantic_cache[key], "cache_hit": True}
    result = compute_fn(c)
    _semantic_cache[key] = result
    return {**result, "cache_hit": False}

**Resultado esperado:** sem output direto — o efeito aparece na seção 7.

### 3. Agent Analyzer — real com fallback (Semana 1, Extended Thinking S7)

**Por que esta função existe:** é o primeiro dos dois agentes da cadeia
(seção 5 tem o segundo). `call_claude_analyzer` manda as métricas da
campanha pra Claude e pede raciocínio explícito antes da classificação
(*extended thinking*); sem API key, `heuristic_analyzer` aplica os mesmos
limiares, calibrados na distribuição real de conversão do dataset (mediana
~16%, top quartil ~35%+).

In [4]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
USE_REAL_LLM = bool(ANTHROPIC_API_KEY)

if USE_REAL_LLM:
    import anthropic
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

ANALYZER_TOOL_SCHEMA = {
    "name": "record_campaign_analysis",
    "description": "Registra a análise estruturada da campanha",
    "input_schema": {
        "type": "object",
        "properties": {
            "reasoning": {"type": "array", "items": {"type": "string"}},
            "performance": {"type": "string", "enum": ["scale", "optimize", "pause"]},
        },
        "required": ["reasoning", "performance"],
    },
}

def call_claude_analyzer(c: CampaignRecord) -> Optional[dict]:
    if not USE_REAL_LLM:
        return None
    response = _client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        tools=[ANALYZER_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "record_campaign_analysis"},
        messages=[{
            "role": "user",
            "content": (
                "Analise esta campanha de telemarketing bancário passo a passo "
                "(liste seu raciocínio) e classifique a performance:\n"
                f"canal={c.channel}, taxa de conversão={c.conversion_rate:.1%}, "
                f"custo por conversão=${c.cost_per_success}, {c.n_contacts} contatos"
            ),
        }],
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None

def heuristic_analyzer(c: CampaignRecord) -> dict:
    """Fallback — limiares calibrados na distribuição real do dataset."""
    reasoning = []
    if c.conversion_rate < 0.08:
        reasoning.append(f"taxa de conversão baixa ({c.conversion_rate:.1%}) → abaixo da mediana real (~16%)")
    if c.cost_per_success > 100:
        reasoning.append(f"custo por conversão alto (${c.cost_per_success}) → ligações longas sem resultado")
    if c.conversion_rate >= 0.35:
        reasoning.append(f"conversão forte ({c.conversion_rate:.1%}) → no top quartil real → candidata a scale up")

    performance: Literal["scale", "optimize", "pause"]
    if c.conversion_rate >= 0.35:
        performance = "scale"
    elif c.conversion_rate < 0.08:
        performance = "pause"
    else:
        performance = "optimize"
    return {"reasoning": reasoning or ["performance dentro da faixa esperada"], "performance": performance}

def analyzer_agent(c: CampaignRecord) -> dict:
    result = call_claude_analyzer(c) or heuristic_analyzer(c)
    return {"campaign_id": c.campaign_id, **result}

print(f"🔑 Modo: {'API REAL (Claude Haiku)' if USE_REAL_LLM else 'MOCK — defina ANTHROPIC_API_KEY pra usar a API real'}")

🔑 Modo: MOCK — defina ANTHROPIC_API_KEY pra usar a API real


**Resultado esperado:** `🔑 Modo: MOCK — ...` (sem chave configurada).

### 4. Tools em paralelo (Semana 5) — 2-5x mais rápido que sequencial

**Por que esta função existe:** demonstra a diferença entre chamar duas
tools uma depois da outra vs simultaneamente. As duas funções simulam
sistemas internos reais de um banco (benchmark de conversão por canal,
teto de contatos permitido por compliance); `gather_context_parallel` roda
as duas ao mesmo tempo com `asyncio.gather`.

In [5]:
import asyncio

async def tool_fetch_benchmark(channel: str) -> dict:
    await asyncio.sleep(0.05)  # simula latência de sistema interno
    benchmarks = {"cellular": 0.18, "telephone": 0.14}  # calibrado na mediana real por canal
    return {"channel": channel, "benchmark_conversion": benchmarks[channel]}

async def tool_fetch_contact_cap(campaign_id: str) -> dict:
    await asyncio.sleep(0.05)
    return {"campaign_id": campaign_id, "max_contacts_per_day": 500}  # limite de compliance

async def gather_context_parallel(c: CampaignRecord) -> dict:
    benchmark, cap = await asyncio.gather(
        tool_fetch_benchmark(c.channel),
        tool_fetch_contact_cap(c.campaign_id),
    )
    return {"benchmark": benchmark, "cap": cap}

**Resultado esperado:** sem output — chamada dentro do grafo (seção 6).

### 5. Agent Recommender — segundo agente da cadeia multi-agent

**Por que esta classe existe:** separar Analyzer (o quê está acontecendo) de
Recommender (o quê fazer sobre isso) é o padrão *multi-agent* da Semana 5:
cada agente tem um prompt/responsabilidade mais estreita.

In [6]:
def recommender_agent(analysis: dict, context: dict) -> dict:
    action_map = {
        "scale": f"Aumentar volume de contatos (teto de compliance: {context['cap']['max_contacts_per_day']}/dia)",
        "optimize": "Revisar script do operador + focar em horários com melhor conversão",
        "pause": "Pausar canal e realocar equipe pro canal com melhor conversão",
    }
    return {
        "campaign_id": analysis["campaign_id"],
        "action": action_map[analysis["performance"]],
        "vs_benchmark": "acima" if analysis["performance"] == "scale" else "abaixo/na média",
    }

### 6. Grafo com branch condicional (Semana 6-7)

**Por que esta estrutura existe:** o `add_conditional_edges` é o que torna
esse grafo diferente do linear do Projeto #1 — a função
`route_by_performance` decide dinamicamente o próximo nó.

In [7]:
from langgraph.graph import StateGraph, START, END

class CampaignState(BaseModel):
    campaign: CampaignRecord
    context: Optional[dict] = None
    analysis: Optional[dict] = None
    recommendation: Optional[dict] = None
    latency_ms: float = 0

    model_config = {"arbitrary_types_allowed": True}

def node_gather(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.context = asyncio.run(gather_context_parallel(state.campaign))
    state.latency_ms += (time.time() - t0) * 1000
    return state

def node_analyze(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.analysis = cached_analysis(state.campaign, analyzer_agent)
    state.latency_ms += (time.time() - t0) * 1000
    return state

def route_by_performance(state: CampaignState) -> str:
    """Branch condicional: estrutura pronta pra rotear diferente por
    performance (ex.: 'pause' indo pra um nó de auditoria de compliance)."""
    return "recommend"

def node_recommend(state: CampaignState) -> CampaignState:
    state.recommendation = recommender_agent(state.analysis, state.context)
    return state

graph = StateGraph(CampaignState)
graph.add_node("gather", node_gather)
graph.add_node("analyze", node_analyze)
graph.add_node("recommend", node_recommend)
graph.add_edge(START, "gather")
graph.add_edge("gather", "analyze")
graph.add_conditional_edges("analyze", route_by_performance, {"recommend": "recommend"})
graph.add_edge("recommend", END)

marketing_agent = graph.compile()

### 7. Rodando + observability de latência (Semana 8)

In [8]:
for c in campaigns:
    result = marketing_agent.invoke(CampaignState(campaign=c))
    rec = result["recommendation"] if isinstance(result, dict) else result.recommendation
    lat = result["latency_ms"] if isinstance(result, dict) else result.latency_ms
    cache_hit = result["analysis"]["cache_hit"] if isinstance(result, dict) else result.analysis["cache_hit"]
    print(f"→ {c.campaign_id} [{c.channel}]: conv={c.conversion_rate:.1%} → {rec['action']} "
          f"(latência={lat:.1f}ms, cache_hit={cache_hit})")

→ cellular_apr [cellular]: conv=20.4% → Revisar script do operador + focar em horários com melhor conversão (latência=65.6ms, cache_hit=False)
→ cellular_aug [cellular]: conv=10.4% → Revisar script do operador + focar em horários com melhor conversão (latência=58.8ms, cache_hit=False)
→ cellular_dec [cellular]: conv=52.8% → Aumentar volume de contatos (teto de compliance: 500/dia) (latência=61.2ms, cache_hit=False)
→ cellular_jul [cellular]: conv=9.9% → Revisar script do operador + focar em horários com melhor conversão (latência=60.7ms, cache_hit=False)
→ cellular_jun [cellular]: conv=42.1% → Aumentar volume de contatos (teto de compliance: 500/dia) (latência=59.9ms, cache_hit=False)
→ cellular_mar [cellular]: conv=51.7% → Aumentar volume de contatos (teto de compliance: 500/dia) (latência=60.8ms, cache_hit=True)
→ cellular_may [cellular]: conv=11.3% → Revisar script do operador + focar em horários com melhor conversão (latência=62.4ms, cache_hit=True)
→ cellular_nov [cellular]: conv=

**Resultado esperado:** 20 linhas, uma por campanha real (canal × mês),
mostrando a taxa de conversão real e a ação recomendada — dá pra ver os
grupos de março/dezembro (conversão real alta, ~50-64%) recebendo "scale",
e os grupos de meio de ano (conversão mais baixa) recebendo "optimize" ou
"pause".

### 8. Testes básicos (Semana 9)

In [9]:
def test_conversion_rate_in_valid_range():
    for c in campaigns:
        assert 0 <= c.conversion_rate <= 1
    print("✓ test_conversion_rate_in_valid_range passou")

def test_recommendation_matches_performance():
    high_conv = max(campaigns, key=lambda c: c.conversion_rate)
    result = marketing_agent.invoke(CampaignState(campaign=high_conv))
    analysis = result["analysis"] if isinstance(result, dict) else result.analysis
    rec = result["recommendation"] if isinstance(result, dict) else result.recommendation
    if analysis["performance"] == "scale":
        assert "Aumentar" in rec["action"]
    print("✓ test_recommendation_matches_performance passou")

test_conversion_rate_in_valid_range()
test_recommendation_matches_performance()

✓ test_conversion_rate_in_valid_range passou
✓ test_recommendation_matches_performance passou


**Resultado esperado:** 2 linhas `✓ ... passou`.

**Próximos passos pra produção:**
- Já dá pra usar Claude de verdade — só definir `ANTHROPIC_API_KEY`
- Semantic cache real com embeddings (Vertex AI Embeddings + Redis)
- Custo real por ligação (hoje é estimado a $2/min) integrado do sistema financeiro do banco
- Avaliação real: comparar a recomendação com o resultado do próximo ciclo de campanha